
LOGESHWARAN J
GroupDNA
29-06-2026

In [3]:
import numpy as np
from datetime import datetime, timedelta
import string

file_path = 'hostel_bois.txt'
media_tag = "<Media omitted>"
deleted_tag = "This message was deleted"

stop_words = {
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it',
    'that', 'this', 'you', 'my', 'me', 'with', 'are', 'was', 'have', 'but',
    'so', 'not', 'be', 'we', 'they', 'at', 'what', 'hai', 'ki', 'ka', 'ko',
    'se', 'tha', 'bhai', 'yaar', 'kya', 'scene', 'guys', 'bhaiya', 'haan', 'abey'
}

caring_keywords = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
comedy_keywords = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

# FEATURE 1: THE CHAT PARSER

messages = []
system_msgs_count = 0
media_msgs_count = 0
deleted_msgs_count = 0
participants = set()

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for line in lines:
    line = line.strip()
    if not line:
        continue

    if ' - ' not in line:
        continue

    dt_str, rest = line.split(' - ', 1)

    try:
        dt_obj = datetime.strptime(dt_str, "%d/%m/%y, %H:%M")
    except ValueError:
        continue

    if ': ' not in rest:
        system_msgs_count += 1
        continue

    sender, text = rest.split(': ', 1)
    sender = sender.strip()
    participants.add(sender)

    is_real = True
    is_deleted = False

    if text == media_tag:
        media_msgs_count += 1
        is_real = False
    elif text == deleted_tag:
        deleted_msgs_count += 1
        is_real = False
        is_deleted = True

    messages.append({
        'timestamp': dt_obj,
        'date_str': dt_obj.strftime('%d/%m/%y'),
        'hour': dt_obj.hour,
        'sender': sender,
        'text': text,
        'is_real': is_real,
        'is_deleted': is_deleted
    })

participant_list = sorted(list(participants))

# FEATURE 2 & 3: STATS PROCESSING

total_messages = len(messages)
first_date = messages[0]['timestamp']
last_date = messages[-1]['timestamp']
total_days = (last_date - first_date).days + 1
period_str = f"{first_date.strftime('%d %B %Y')} to {last_date.strftime('%d %B %Y')}"

person_counts = {}
for m in messages:
    person_counts[m['sender']] = person_counts.get(m['sender'], 0) + 1
sorted_person_counts = sorted(person_counts.items(), key=lambda x: x[1], reverse=True)

date_counts = {}
hour_counts = {}
for m in messages:
    d = m['date_str']
    h = m['hour']
    date_counts[d] = date_counts.get(d, 0) + 1
    hour_counts[h] = hour_counts.get(h, 0) + 1

busiest_day_str = max(date_counts, key=date_counts.get)
busiest_day_dt = datetime.strptime(busiest_day_str, '%d/%m/%y')
busiest_hour = max(hour_counts, key=hour_counts.get)

# FEATURE 4: ACTIVITY HEATMAP (NUMPY)

heatmap_matrix = np.zeros((len(participant_list), 24), dtype=int)
for m in messages:
    p_idx = participant_list.index(m['sender'])
    h_idx = m['hour']
    heatmap_matrix[p_idx, h_idx] += 1

# FEATURE 5: WORD FREQUENCY (TOP WORDS)

word_counts = {}
for m in messages:
    if not m['is_real']:
        continue
    clean_text = m['text'].lower().translate(str.maketrans('', '', string.punctuation))
    for word in clean_text.split():
        if word not in stop_words and len(word) > 1:
            word_counts[word] = word_counts.get(word, 0) + 1

top_10_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

# FEATURE 6: RESPONSE PATTERNS & STREAKS

response_gaps = {p: [] for p in participant_list}
active_days_per_person = {p: set() for p in participant_list}
last_time = None
last_sender = None

for m in messages:
    sender = m['sender']
    dt = m['timestamp']
    active_days_per_person[sender].add(m['date_str'])

    if last_sender and last_sender != sender:
        gap_seconds = (dt - last_time).total_seconds()
        response_gaps[sender].append(gap_seconds)

    last_time = dt
    last_sender = sender

avg_responses = {}
for p in participant_list:
    gaps = response_gaps[p]
    avg_responses[p] = sum(gaps) / len(gaps) if gaps else 0

fastest_replier = min([p for p in avg_responses if avg_responses[p] > 0], key=avg_responses.get)
slowest_replier = max(avg_responses, key=avg_responses.get)

all_calendar_days = []
current_day = first_date
while current_day <= last_date:
    all_calendar_days.append(current_day.strftime('%d/%m/%y'))
    current_day += timedelta(days=1)

longest_silent_streaks = {}
for p in participant_list:
    max_streak, current_streak = 0, 0
    for day in all_calendar_days:
        if day not in active_days_per_person[p]:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0
    longest_silent_streaks[p] = max_streak

# FEATURE 7: PERSONALITY ARCHETYPES

metrics = {p: {
    'total_msgs': 0, 'real_msgs': 0, 'night_msgs': 0, 'total_words': 0,
    'caring_score': 0, 'drama_msgs': 0, 'comedy_msgs': 0, 'question_msgs': 0,
    'bursts': [], 'silent_days': total_days - len(active_days_per_person[p])
} for p in participant_list}

curr_sender = None
curr_burst = 0

for m in messages:
    p = m['sender']
    metrics[p]['total_msgs'] += 1

    if m['hour'] >= 23 or m['hour'] < 5:
        metrics[p]['night_msgs'] += 1

    if p == curr_sender:
        curr_burst += 1
    else:
        if curr_sender:
            metrics[curr_sender]['bursts'].append(curr_burst)
        curr_sender = p
        curr_burst = 1

    if m['is_real']:
        metrics[p]['real_msgs'] += 1
        text = m['text']
        text_lower = text.lower()

        metrics[p]['total_words'] += len(text.split())
        metrics[p]['caring_score'] += sum(1 for kw in caring_keywords if kw in text_lower)
        metrics[p]['comedy_msgs'] += sum(1 for kw in comedy_keywords if kw in text_lower)

        if (text.isupper() and len(text) >= 3) or text.count('!') >= 2:
            metrics[p]['drama_msgs'] += 1

        if text.strip().endswith('?'):
            metrics[p]['question_msgs'] += 1

if curr_sender:
    metrics[curr_sender]['bursts'].append(curr_burst)

assigned_archetypes = {}
mom_candidate = max(participant_list, key=lambda x: metrics[x]['caring_score'])
assigned_archetypes[mom_candidate] = ("THE GROUP MOM", f"caring keyword score: {metrics[mom_candidate]['caring_score']}")

for p in participant_list:
    if p in assigned_archetypes:
        continue

    s = metrics[p]
    real_count = s['real_msgs'] if s['real_msgs'] > 0 else 1
    total_count = s['total_msgs']
    avg_burst = sum(s['bursts']) / len(s['bursts']) if s['bursts'] else 0
    silent_percentage = (s['silent_days'] / total_days) * 100
    night_percentage = (s['night_msgs'] / total_count) * 100
    avg_words = s['total_words'] / real_count
    drama_percentage = (s['drama_msgs'] / real_count) * 100

    if silent_percentage > 60:
        assigned_archetypes[p] = ("THE GHOST", f"silent on {s['silent_days']} of {total_days} days")
    elif avg_burst > 3:
        assigned_archetypes[p] = ("THE SPAMMER", f"avg {avg_burst:.1f} msgs in a row")
    elif night_percentage > 60:
        assigned_archetypes[p] = ("THE NIGHT OWL", f"{night_percentage:.1f}% msgs between 23h-04h")
    elif avg_words > 30:
        assigned_archetypes[p] = ("THE STORYTELLER", f"avg {avg_words:.1f} words per msg")
    elif drama_percentage > 30:
        assigned_archetypes[p] = ("THE DRAMA QUEEN", f"{drama_percentage:.1f}% ALL-CAPS messages")
    else:
        assigned_archetypes[p] = ("THE COMEDIAN", f"{(s['comedy_msgs']/real_count)*100:.1f}% custom vocabulary")


# FEATURE 8: THE UNIFIED FINAL REPORT

print("============================================================")
print("                      GROUPDNA REPORT                       ")
print("============================================================")
print(f"Group Chat   : \"Hostel Bois 4ever\"")
print(f"Period       : {period_str} ({total_days} days)")
print(f"Busiest Day  : {busiest_day_dt.strftime('%d %B %Y')} ({date_counts[busiest_day_str]} messages)")
print(f"Busiest Hour : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00")
print(f"Total Lines  : {total_messages:,} valid timeline logs")
print(f"Analytics    : Skipped {system_msgs_count} system, {media_msgs_count} media files, {deleted_msgs_count} deleted logs.")
print("------------------------------------------------------------")

print("\nMESSAGES PER PERSON")
print("------------------------------------------------------------")
for person, count in sorted_person_counts:
    pct = (count / total_messages) * 100
    print(f" {person:<12} : {count:>5} messages ({pct:.1f}%)")

print("\nACTIVITY HEATMAP (hour of day, columns 00 to 23)")
print("------------------------------------------------------------")
print("             00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23")
for i, person in enumerate(participant_list):
    row_values = heatmap_matrix[i]
    max_val = row_values.max() if row_values.max() > 0 else 1
    shading_line = ""
    for val in row_values:
        ratio = val / max_val
        if ratio == 0:     shading_line += " . "
        elif ratio <= 0.3: shading_line += " ░ "
        elif ratio <= 0.6: shading_line += " ▒ "
        else:              shading_line += " █ "
    print(f" {person:<11} {shading_line}")

print("\nTHIS GROUP'S TOP DATA ANALYSIS WORDS")
print("------------------------------------------------------------")
max_word_frequency = top_10_words[0][1] if top_10_words else 1
for word, count in top_10_words:
    bar_length = int((count / max_word_frequency) * 25)
    bar_graphic = "█" * bar_length
    print(f" {word:<12} : {count:>4} {bar_graphic}")

print("\nRESPONSE SPEED INTERACTION PATTERNS")
print("------------------------------------------------------------")
print(f" Fastest replier : {fastest_replier:<10} (avg {avg_responses[fastest_replier]/60:.1f} minutes)")
print(f" Slowest replier : {slowest_replier:<10} (avg {avg_responses[slowest_replier]/3600:.1f} hours)")

print("\nLONGEST SILENT STREAKS")
print("------------------------------------------------------------")
for person, streak in sorted(longest_silent_streaks.items(), key=lambda x: x[1], reverse=True):
    streak_desc = f"{streak} days" if streak > 0 else "0 days (never went silent)"
    print(f" {person:<12} : {streak_desc}")

print("\nPERSONALITY ARCHETYPES DETECTED")
print("------------------------------------------------------------")
for person in participant_list:
    arch_title, arch_metric = assigned_archetypes[person]
    print(f" {person:<12} → {arch_title:<18} ({arch_metric})")

print("============================================================")
print(" Generated by GroupDNA                Built with Python + NumPy")
print("============================================================")

                      GROUPDNA REPORT                       
Group Chat   : "Hostel Bois 4ever"
Period       : 01 April 2024 to 30 May 2024 (60 days)
Busiest Day  : 04 May 2024 (76 messages)
Busiest Hour : 18:00 - 19:00
Total Lines  : 3,174 valid timeline logs
Analytics    : Skipped 4 system, 32 media files, 15 deleted logs.
------------------------------------------------------------

MESSAGES PER PERSON
------------------------------------------------------------
 Rahul        :   953 messages (30.0%)
 Priya        :   718 messages (22.6%)
 Neha         :   635 messages (20.0%)
 Aman         :   490 messages (15.4%)
 Karan        :   354 messages (11.2%)
 Vikas        :    24 messages (0.8%)

ACTIVITY HEATMAP (hour of day, columns 00 to 23)
------------------------------------------------------------
             00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 Aman         █  █  █  █  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  █ 
 Karan        .